In [18]:
import xtrack as xt

## Loading lattices and strengths

For more info on lattice import options, see corresponding [sections in the User's guide](https://xsuite.readthedocs.io/en/latest/environment.html#saving-and-loading-environment-or-individual-lines).

Three main possibilities to define and save beam line lattices:
 - Lattice defined as **Python** script
 - Lattice defined as **JSON** file
 - Lattice defined as **MAD-X** script

### Lattice as Python script

In the previous notebook we have built a lattice in a Python notebook.
We can simply save that code as a `.py` script and it becomes a lattice file:

<div style="max-height: 200px;
            overflow-y: auto;
            padding: 10px;
            border: 1px solid #ccc;
            font-size: 10px;">

```
## pimm_seq.py

import xtrack as xt
import numpy as np

env = xt.get_environment()
env.particle_ref = xt.Particles(kinetic_energy0=200e6)
env.vars.default_to_zero = True

############
# Elements #
############

# Element geometry
n_bends = 16
env['ang_mb'] = 2*np.pi/n_bends
env['l_mb'] = 1.65
env['l_mq'] = 0.35

# Magnet types
env.new('mb', xt.RBend, length_straight='l_mb', angle='ang_mb')
env.new('mq', xt.Quadrupole, length='l_mq')
env.new('ms', xt.Sextupole, length=0.2)

# Quadrupole families
env.new('qfa', 'mq', k1= 'kqfa')
env.new('qfb', 'mq', k1= 'kqfb')
env.new('qd',  'mq', k1= 'kqd')

# Magnet instances
env.new('msf.1', 'ms', k2='ksf')
env.new('msf.2', 'ms', k2='ksf')
env.new('msd.1', 'ms', k2='ksd')
env.new('msd.2', 'ms', k2='ksd')
env.new('mse',   'ms', k2='kse')

# RF cavity
env.new('rf1', xt.Cavity, voltage='vrf', frequency='frf')

###########
# Lattice #
###########

# Cells
cell_a = env.new_line(length=7.405, components=[
    env.place('qfa', at=0.3875),
    env.place('mb', at=1.8125),
    env.place('qd', at=3.2925),
    env.place('mb', at=5.0475),
    env.place('qfa', at=6.3275),
])

cell_b = env.new_line(name='cell_b', length=8.405, components=[
    env.place('qfb', at=1.2725),
    env.place('mb', at= 2.7275),
    env.place('qd', at=4.8575),
    env.place('mb', at=6.5125),
    env.place('qfb', at=7.7925),
])

# Arc
arc = cell_a + cell_b

# Straight sections
long_straight = env.new_line(length=2., components=[
    env.new('mid.lss', xt.Marker, at=1.)
])
short_straight = env.new_line(length=1., components=[
    env.new('mid.sss', xt.Marker, at=1.)
])

# Ring
ring = 2 * (long_straight
              + arc
             + short_straight
             - arc # mirror symmetric lattice
            )

# Assign unique names to all elements
ring.replace_all_repeated_elements()

# Insert sextupoles
ring.insert([
    env.place('msf.1', at=-0.2, from_='qfb.0@start'),
    env.place('msf.2', at=-0.2, from_='qfb.4@start'),
    env.place('msd.1', at=0.3,  from_='qd.2@end'),
    env.place('msd.2', at=0.3,  from_='qd.6@end'),
    env.place('mse',   at=-0.3, from_='qfa.4@start')
])

# Insert RF
ring.insert('rf1', at=0.5, from_='qfa.3@start')

# Select lines to keep
env['ring'] = ring

env.vars.default_to_zero = False
```
</div>

In [19]:
%%capture

# The file can be executed or imported through Python or loaded with `xt.load(...)`
env = xt.load('./pimm_seq.py')

In [20]:
env

Environment(2 lines: {cell_b, ring}, 84 elements, 12 vars, 0 particles)

### Lattice as JSON file
The `xt.load(...)` function can be used to load JSON files saved by Xsuite

In [21]:
env = xt.load('./pimm.json')

Loading line from dict:   0%|          | 0/86 [00:00<?, ?it/s]

In [22]:
env

Environment(2 lines: {cell_a, ring}, 86 elements, 12 vars, 0 particles)

### Lattice as MAD-X file
The `xt.load(...)` function can also load MAD-X files (when they contain only definitions of variables, elements, sequences).
This does not execute MAD-X, the file is loaded by the Xsuite parser.

In [23]:
env = xt.load('./PIMM.seq')

In [24]:
env

Environment(1 lines: {ring}, 99 elements, 24 vars, 0 particles)

### Optics files

It is possible to have different optics files for the same lattice (files containing only variables - numbers or expressions)

Optics as a JSON file (`example_strengths.json`):
<div style="max-height: 300px;
            overflow-y: auto;
            padding: 10px;
            border: 1px solid #ccc;
            font-size: 12px;">
    
```json
{
 "kqd": -0.5904781382942978,
 "kqfa": 0.3377330649315182,
 "kqfb": 0.546941967560322,
 "ksd": -0.7807717051055789,
 "ksf": 0.5835018163902821,
 "myvar": "2 * kqfa"
}
```

</div>


Optics as a MAD-X file (`example_strengths.madx`):
<div style="max-height: 300px;
            overflow-y: auto;
            padding: 10px;
            border: 1px solid #ccc;
            font-size: 12px;">

```    
! Quadrupoles
kqd = -0.5904781382942978;
kqfa = 0.3377330649315182;
kqfb = 0.546941967560322;

! Sextupoles
ksd = -0.7807717051055789;
ksf = 0.5835018163902821;

! A knob (deferred expression)
myvar := 2 * kqfa;
```

</div>

They can be loaded directly in the vars container of the environment:

In [25]:
env.vars.load('example_strengths.json')

In [26]:
env.vars.load('example_strengths.madx')

In [27]:
env.info('myvar')

Info for vars['myvar']

value: 0.6754661298630364

controlled by expr:
  vars['myvar'] = (2.0 * vars['kqfa'])

expr_dependencies:
  vars['kqfa'] = 0.3377330649315182

controlled_targets: None

